# **1. 라이브러리 불러오기**

In [ ]:
# If running in Colab, mount Google Drive here.
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
import numpy as np
import pandas as pd
import gc
import matplotlib.pyplot as plt

from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# **2. 데이터 불러오기 및 간단 EDA**

In [ ]:
train=pd.read_csv('../data/train.csv')
vali = pd.read_csv('../data/validation_sample.csv')
test=pd.read_csv('../data/test.csv')
submission=pd.read_csv('../data/sample_submission.csv')

In [ ]:
print(train.shape, test.shape, vali.shape, submission.shape)

In [ ]:
train.head()

In [ ]:
#test는 level값이 없습니다
test.head()

In [ ]:
train_df = train
test_df = test

In [ ]:
first_strings = pd.merge(train_df.loc[train_df.level!=0].full_log.apply(lambda x: x.split(' ')[0]).value_counts().rename('lv!=0'),
                         train_df.loc[train_df.level==0].full_log.apply(lambda x: x.split(' ')[0]).value_counts().rename('lv==0'),
                         how='outer',left_index=True,right_index=True).fillna(0).astype(int)

first_strings = first_strings.sort_values('lv==0',ascending=False)

In [ ]:
first_strings

In [ ]:
train['full_log'].value_counts()

In [ ]:
train['level'].value_counts()

In [ ]:
# text 전처리로 문자만 남겨 보았다.
train['full_log'] = train['full_log'].str.replace(pat=r'[^A-Za-z]', repl=r'', regex=True)
test['full_log'] = test['full_log'].str.replace(pat=r'[^A-Za-z]', repl=r'', regex=True)

In [ ]:
#full_log에서 숫자는 마스킹 처리
train['full_log']=train['full_log'].str.replace(r'[0-9]', ' ')
test['full_log']=test['full_log'].str.replace(r'[0-9]', ' ')

In [ ]:
train['full_log'].value_counts()

In [ ]:
train['full_log'].value_counts()

In [ ]:
#train['full_log'] => train_text로 list
#train['level']=> train_level로 array
train_text=list(train['full_log'])
train_level=np.array(train['level'])

In [ ]:
text_train, text_val, label_train, label_val = train_test_split(train['full_log'], train['level'], test_size=0.2, random_state=42, stratify=train['level'])

In [ ]:
import tensorflow as tf

tokenizer = tf.keras.preprocessing.text.Tokenizer()
tokenizer.fit_on_texts(text_train)
top_k = len(tokenizer.word_index)

x_train = tokenizer.texts_to_sequences(text_train)
x_val = tokenizer.texts_to_sequences(text_val)

max_length=100

x_train_vector = tf.keras.preprocessing.sequence.pad_sequences(
    x_train, maxlen=max_length, padding='post'
)

x_val_vector = tf.keras.preprocessing.sequence.pad_sequences(
    x_val, maxlen=max_length, padding='post'
)

In [ ]:
x_train_vector.shape

In [ ]:
x_val_vector.shape

In [ ]:
x_train_vector

In [ ]:
# # 검증 값
from sklearn import metrics

# def macro_f1(answer_df, submission_df):

#     submission_df = submission_df[submission_df['id'].isin(answer_df['id'])]

#     submission_df.index = range(submission_df.shape[0])

    

#     true = answer_df['level']

#     pred = submission_df['level']

    

#     score = metrics.f1_score(y_true=true, y_pred=pred, average='macro')

    

#     return score

In [ ]:
def macro_f1(answer_df, submission_df):    
    true = answer_df
    pred = submission_df

    score = metrics.f1_score(y_true=true, y_pred=pred, average='macro')
    return score    

In [ ]:
pred

In [ ]:
label_val

In [ ]:
macro_f1(label_val,pred)

# **3. 데이터 전처리**

In [ ]:
#full_log에서 숫자는 마스킹 처리
train['full_log']=train['full_log'].str.replace(r'[0-9]', '<num>')
test['full_log']=test['full_log'].str.replace(r'[0-9]', '<num>')

In [ ]:
#train['full_log'] => train_text로 list
#train['level']=> train_level로 array
train_text=list(train['full_log'])
train_level=np.array(train['level'])

In [ ]:
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tqdm import tqdm

In [ ]:
text_train, text_val, label_train, label_val = train_test_split(train['full_log'], train['level'], test_size=0.2, random_state=42)

In [ ]:
tokenizer = tf.keras.preprocessing.text.Tokenizer()
tokenizer.fit_on_texts(text_train)
top_k = len(tokenizer.word_index)

x_train = tokenizer.texts_to_sequences(text_train)
x_val = tokenizer.texts_to_sequences(text_val)

max_length=100

x_train_vector = tf.keras.preprocessing.sequence.pad_sequences(
    x_train, maxlen=max_length, padding='post'
)

x_val_vector = tf.keras.preprocessing.sequence.pad_sequences(
    x_val, maxlen=max_length, padding='post'
)

In [ ]:
x_train_vector.shape

In [ ]:
x_val_vector.shape

In [ ]:
x_train_vector

In [ ]:
#랜덤포레스트로 모델링
from sklearn.ensemble import RandomForestClassifier

forest=RandomForestClassifier(n_estimators=1000, random_state=42)

forest.fit(x_train_vector, label_train)

* https://injo.tistory.com/44

In [ ]:
# LGBM 모델
lgb = LGBMClassifier(n_estimators=400, random_state=42)
evals = [(x_val_vector, label_val)]
lgb.fit(x_train_vector, label_train, early_stopping_rounds=50, eval_metric='error', eval_set=evals, verbose=True)

In [ ]:
# LGBM 모델
import xgboost as xgb ## XGBoost 불러오기
lgb = xgb.XGBClassifier(n_estimators=1000, random_state=42)
evals = [(x_val_vector, label_val)]
lgb.fit(x_train_vector, label_train, early_stopping_rounds=50, eval_metric='merror', eval_set=evals, verbose=True)

* XGBClassifier
              (base_score=0.5, booster='gbtree', colsample_bylevel=1,
              colsample_bynode=1, colsample_bytree=1, gamma=0,
              learning_rate=0.1, max_delta_step=0, max_depth=3,
              min_child_weight=1, missing=None, n_estimators=1000, n_jobs=1,
              nthread=None, objective='multi:softprob', random_state=42,
              reg_alpha=0, reg_lambda=1, scale_pos_weight=1, seed=None,
              silent=None, subsample=1, verbosity=1)

* 0.9472021192302403

In [ ]:
# LGBM 모델
import xgboost as xgb ## XGBoost 불러오기
lgb = xgb.XGBClassifier(n_estimators=1500, random_state=42, n_jobs=-1)
evals = [(x_val_vector, label_val)]
lgb.fit(x_train_vector, label_train, early_stopping_rounds=50, eval_metric='merror', eval_set=evals, verbose=True)

* XGBClassifier
              (base_score=0.5, booster='gbtree', colsample_bylevel=1,
              colsample_bynode=1, colsample_bytree=1, gamma=0,
              learning_rate=0.1, max_delta_step=0, max_depth=3,
              min_child_weight=1, missing=None, n_estimators=1500, n_jobs=1,
              nthread=None, objective='multi:softprob', random_state=42,
              reg_alpha=0, reg_lambda=1, scale_pos_weight=1, seed=None,
              silent=None, subsample=1, verbosity=1)

* 0.9472021192302403

In [ ]:
# LGBM 모델
lgb = LGBMClassifier(n_estimators=400, random_state=42)
evals = [(x_val_vector, label_val)]
lgb.fit(x_train_vector, label_train, early_stopping_rounds=50, eval_metric='error', eval_set=evals, verbose=True)

In [ ]:
# LGBM 모델
import xgboost as xgb ## XGBoost 불러오기
lgb = xgb.XGBClassifier(n_estimators=1000, random_state=42)
evals = [(x_val_vector, label_val)]
lgb.fit(x_train_vector, label_train, early_stopping_rounds=50, eval_metric='merror', eval_set=evals, verbose=True)

* XGBClassifier
              (base_score=0.5, booster='gbtree', colsample_bylevel=1,
              colsample_bynode=1, colsample_bytree=1, gamma=0,
              learning_rate=0.1, max_delta_step=0, max_depth=3,
              min_child_weight=1, missing=None, n_estimators=1000, n_jobs=1,
              nthread=None, objective='multi:softprob', random_state=42,
              reg_alpha=0, reg_lambda=1, scale_pos_weight=1, seed=None,
              silent=None, subsample=1, verbosity=1)

* 0.9472021192302403

In [ ]:
# LGBM 모델
import xgboost as xgb ## XGBoost 불러오기
lgb = xgb.XGBClassifier(n_estimators=1500, random_state=42, n_jobs=-1)
evals = [(x_val_vector, label_val)]
lgb.fit(x_train_vector, label_train, early_stopping_rounds=50, eval_metric='merror', eval_set=evals, verbose=True)

* XGBClassifier
              (base_score=0.5, booster='gbtree', colsample_bylevel=1,
              colsample_bynode=1, colsample_bytree=1, gamma=0,
              learning_rate=0.1, max_delta_step=0, max_depth=3,
              min_child_weight=1, missing=None, n_estimators=1500, n_jobs=1,
              nthread=None, objective='multi:softprob', random_state=42,
              reg_alpha=0, reg_lambda=1, scale_pos_weight=1, seed=None,
              silent=None, subsample=1, verbosity=1)

* 0.9472021192302403

In [ ]:
# a = [100, 200, 300, 400, 600, 800]

# for i in tqdm(a):
#   print(i)
#   forest=RandomForestClassifier(n_estimators=i)
#   forest.fit(x_train_vector, label_train)

#   preds=forest.predict(x_val_vector)
#   f1 = macro_f1(label_val,preds)
#   print(f1)

In [ ]:
x_train_vector

In [ ]:
x_val_vector

In [ ]:
# 모델 자체
forest.score(x_val_vector, label_val)

In [ ]:
#crosstab으로 확인
pred=forest.predict(x_val_vector)
crosstab = pd.crosstab(label_val, pred, rownames=['real'], colnames=['pred'])
crosstab

In [ ]:
preds=forest.predict(x_val_vector)
probas=forest.predict_proba(x_val_vector)
print(preds.shape)
print(probas.shape)

In [ ]:
from collections import Counter
Counter(label_val)

In [ ]:
# 7을 추가
# preds[np.where(np.max(probas, axis=1)<0.9)]=7
# new_crosstab = pd.crosstab(label_val, preds, rownames=['real'], colnames=['pred'])
# new_crosstab

In [ ]:
macro_f1(label_val,preds)

In [ ]:
#crosstab으로 확인
pred=lgb.predict(x_val_vector)
crosstab = pd.crosstab(label_val, pred, rownames=['real'], colnames=['pred'])
crosstab

In [ ]:
preds=lgb.predict(x_val_vector)
probas=lgb.predict_proba(x_val_vector)
print(preds.shape)
print(probas.shape)

In [ ]:
from collections import Counter
Counter(label_val)

In [ ]:
macro_f1(label_val,preds)

In [ ]:
label_val

In [ ]:
get_clf_eval(label_val,preds)

In [ ]:
# 혼동행렬, 정확도, 정밀도, 재현율, F1, AUC 불러오기
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score

def get_clf_eval(y_test, y_pred):
    confusion = confusion_matrix(y_test, y_pred)
    accuracy = accuracy_score(y_test, y_pred)

    precision = precision_score(y_test, y_pred, average='micro')
    recall = recall_score(y_test, y_pred, average='micro')

    F1 = f1_score(y_test, y_pred, average='micro')

    print('오차행렬:\n', confusion)
    print('\n정확도: {:.4f}'.format(accuracy))
    print('정밀도: {:.4f}'.format(precision))
    print('재현율: {:.4f}'.format(recall))
    print('F1: {:.4f}'.format(F1))

# 다시 시작

In [ ]:
train=pd.read_csv('../data/train.csv')
vali = pd.read_csv('../data/validation_sample.csv')
test=pd.read_csv('../data/test.csv')
submission=pd.read_csv('../data/sample_submission.csv')

In [ ]:
# text 전처리로 문자만 남겨 보았다.
train['full_log'] = train['full_log'].str.replace(r'[0-9,Jan,Feb,Dec,Sep,Oct,Nov,Mar]', '')
test['full_log'] = test['full_log'].str.replace(r'[0-9,Jan,Feb,Dec,Sep,Oct,Nov,Mar]', '')

In [ ]:
train['full_log'].value_counts()

In [ ]:
text_train, text_val, label_train, label_val = train_test_split(train['full_log'], train['level'], test_size=0.2, random_state=42, stratify=train['level'])

In [ ]:
import tensorflow as tf

tokenizer = tf.keras.preprocessing.text.Tokenizer()
tokenizer.fit_on_texts(text_train)
top_k = len(tokenizer.word_index)

x_train = tokenizer.texts_to_sequences(text_train)
x_val = tokenizer.texts_to_sequences(text_val)

max_length=100

x_train_vector = tf.keras.preprocessing.sequence.pad_sequences(
    x_train, maxlen=max_length, padding='post'
)

x_val_vector = tf.keras.preprocessing.sequence.pad_sequences(
    x_val, maxlen=max_length, padding='post'
)

In [ ]:
x_train_vector

* https://statkclee.github.io/model/model-python-xgboost-hyper.html

In [ ]:
# LGBM 모델
import xgboost as xgb ## XGBoost 불러오기
lgb = xgb.XGBClassifier(n_estimators=1000, random_state=42, n_jobs=-1, max_depth= 3)
evals = [(x_val_vector, label_val)]
lgb.fit(x_train_vector, label_train, early_stopping_rounds=50, eval_metric='merror', eval_set=evals, verbose=True)

In [ ]:
#crosstab으로 확인
pred=lgb.predict(x_val_vector)
crosstab = pd.crosstab(label_val, pred, rownames=['real'], colnames=['pred'])
crosstab

In [ ]:
preds=lgb.predict(x_val_vector)
probas=lgb.predict_proba(x_val_vector)
print(preds.shape)
print(probas.shape)

In [ ]:
macro_f1(label_val,preds)

In [ ]:
get_clf_eval(label_val,preds)

In [ ]:
#test 데이터 전처리
text_test = test['full_log'].str.replace(r'[0-9,Jan,Feb,Dec,Sep,Oct,Nov,Mar]', '')

x_test = tokenizer.texts_to_sequences(text_test)

x_test_vector = tf.keras.preprocessing.sequence.pad_sequences(
   x_test , maxlen=max_length, padding='post'
)

In [ ]:
results=lgb.predict(x_test_vector)
results_proba=lgb.predict_proba(x_test_vector)
# results[np.where(np.max(results_proba, axis=1) < 0.9)] = 7

In [ ]:
# results=forest.predict(x_test_vector)
# results_proba=forest.predict_proba(x_test_vector)
# # results[np.where(np.max(results_proba, axis=1) < 0.9)] = 7

In [ ]:
submission['level']=results

In [ ]:
submission

In [ ]:
submission.to_csv('../outputs/submission_lgb2_0512.csv', index=False)

In [ ]:
text_test = test['full_log'].str.replace(r'[0-9]', '<num>')

x_test = tokenizer.texts_to_sequences(text_test)

x_test_vector = tf.keras.preprocessing.sequence.pad_sequences(
   x_test , maxlen=max_length, padding='post'
)

In [ ]:
#CountVectorizer로 벡터화
# from sklearn.feature_extraction.text import CountVectorizer
# vectorizer=CountVectorizer(analyzer="word", max_features=10000)

# train_features=vectorizer.fit_transform(train_text)

In [ ]:
# train_features

In [ ]:
# TfidfVectorizer로 백터화

# from sklearn.feature_extraction.text import TfidfVectorizer
# tfidf_vectorizer = TfidfVectorizer(analyzer="word", max_features=10000) 
# train_features = tfidf_vectorizer.fit_transform(train_text)

In [ ]:
# train_features

> 텍스트 전처리
* https://codetorial.net/tensorflow/natural_language_processing_in_tensorflow_01.html

In [ ]:
# 토큰화랑 시퀸스 변환
import tensorflow as tf
# from tensorflow import keras
# from tensorflow.keras.preprocessing.text import Tokenizer
# from tensorflow.keras.preprocessing.sequence import pad_sequences


# tokenizer = Tokenizer(num_words = 1000, oov_token="<OOV>")
# tokenizer.fit_on_texts(train_text)
# word_index = tokenizer.word_index

# train_features = tokenizer.texts_to_sequences(train_text)
# train_features = pad_sequences(train_features)

# print(word_index)
# print(train_features)

In [ ]:
# train_features

In [ ]:
tokenizer = tf.keras.preprocessing.text.Tokenizer()
tokenizer.fit_on_texts(text_train)
top_k = len(tokenizer.word_index)

x_train = tokenizer.texts_to_sequences(text_train)
x_val = tokenizer.texts_to_sequences(text_val)

max_length=100

x_train_vector = tf.keras.preprocessing.sequence.pad_sequences(
    x_train, maxlen=max_length, padding='post'
)

x_val_vector = tf.keras.preprocessing.sequence.pad_sequences(
    x_val, maxlen=max_length, padding='post'
)

# **4. 모델링**

In [ ]:
#훈련 데이터 셋과 검증 데이터 셋으로 분리
TEST_SIZE=0.2
RANDOM_SEED=42

train_x, eval_x, train_y, eval_y=train_test_split(train_features, train_level, test_size=TEST_SIZE, random_state=RANDOM_SEED)

In [ ]:
#랜덤포레스트로 모델링
from sklearn.ensemble import RandomForestClassifier

forest=RandomForestClassifier(n_estimators=10000)

forest.fit(train_x, train_y)

In [ ]:
#모델 검증
forest.score(eval_x, eval_y)

In [ ]:
#crosstab으로 확인
pred=forest.predict(eval_x)
crosstab = pd.crosstab(eval_y, pred, rownames=['real'], colnames=['pred'])
crosstab

+ 새로운 위험요소에 대한 가정 추가
+ => 예측치의 예측 확률이 0.90이하인 경우, 즉 확신이 없을 경우 이상치로 판단 

In [ ]:
preds=forest.predict(eval_x)
probas=forest.predict_proba(eval_x)
print(preds.shape)
print(probas.shape)

In [ ]:
preds[np.where(np.max(probas, axis=1)<0.9)]=7
new_crosstab = pd.crosstab(eval_y, preds, rownames=['real'], colnames=['pred'])
new_crosstab

# **5. 예측**

In [ ]:
#test 데이터 전처리
test_text=list(test['full_log'])
ids=list(test['id'])

In [ ]:
#test 데이터 vectorizer
#주의: fit_transform의 경우 data leakage에 위반될 수 있습니다
# test_features=vectorizer.transform(test_text)

In [ ]:
# 토큰화랑 시퀸스 설정
test_features = tokenizer.texts_to_sequences(test_text)
test_features = pad_sequences(test_features)
print(test_features)

In [ ]:
results=forest.predict(test_features)
results_proba=forest.predict_proba(test_features)
results[np.where(np.max(results_proba, axis=1) < 0.9)] = 7

In [ ]:
submission['level']=results

In [ ]:
submission

In [ ]:
submission.to_csv('../outputs/submission_0511.csv', index=False)